In [3]:
from ultralytics import YOLO
import cv2
from ultralytics.utils.plotting import Annotator
import os

In [ ]:
# Load YOLO model
killfeed_box_model = YOLO("killfeed/results/run21/weights/best.pt")

# Load video
vod = "whole_vod_useful.mp4"
cap = cv2.VideoCapture(vod)

# Get video properties
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(fps,width,height)

# Define the killfeed region (x, y, width, height.) 
# fThis is the area our model will check for killfeed boxes.
killfeed_x, killfeed_y, killfeed_w, killfeed_h = 850, 50, 430, 350


29 1280 720


Create a killfeed crop of our VOD

In [ ]:
# Define video writer for the cropped region
crop_output_path = "whole_vod_killfeed.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
crop_out = cv2.VideoWriter(crop_output_path, fourcc, fps, (killfeed_w, killfeed_h))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Extract killfeed region from the frame
    killfeed_crop = frame[killfeed_y:killfeed_y+killfeed_h, killfeed_x:killfeed_x+killfeed_w]
    
    # Write cropped region to the new video file
    crop_out.write(killfeed_crop)
    
crop_out.release()

In [6]:
# Define video writer for the cropped region
cap = cv2.VideoCapture(crop_output_path)
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(fps,width,height)

detected_output_path = "processed_useful.mp4"
detected_out = cv2.VideoWriter(detected_output_path, fourcc, fps, (width, height))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = killfeed_box_model(frame, stream=True)
    
    # Optionally, draw bounding boxes on the cropped frame
    for result in results:
        boxes = result.boxes  # Get the boxes from the result
        for box in boxes:
            print(box.xyxy)
            # Example: Draw rectangle for each detected box
            # Assuming box provides x1, y1, x2, y2 coordinates
            x1, y1, x2, y2 = box.xyxy[0]  # need to specify we want the list of coords from the tensor
            
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
    
    if len(result.boxes) > 0:
        # Write the (processed) cropped frame to the new video file
        detected_out.write(frame)
    
    # (Additional processing if needed)
    
detected_out.release()


29 430 350

0: 384x448 3 boxs, 179.0ms
tensor([[  7.8177, 248.7619,  54.5300, 285.2863]])
tensor([[ 49.0223,   0.0000, 392.4622,  23.1086]])
tensor([[  0.0000,   0.0000, 334.6730,  21.1923]])
Speed: 14.9ms preprocess, 179.0ms inference, 18.0ms postprocess per image at shape (1, 3, 384, 448)

0: 384x448 3 boxs, 52.0ms
tensor([[  8.1762, 248.9087,  54.5984, 285.3195]])
tensor([[ 49.6010,   0.0000, 393.1615,  22.8368]])
tensor([[  0.0000,   0.0000, 334.5784,  21.4741]])
Speed: 2.0ms preprocess, 52.0ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 448)

0: 384x448 3 boxs, 52.0ms
tensor([[  8.4114, 248.8236,  54.9545, 285.0438]])
tensor([[ 49.5317,   0.0000, 392.1162,  22.5261]])
tensor([[  0.0000,   0.0000, 334.4843,  21.3770]])
Speed: 1.0ms preprocess, 52.0ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 448)

0: 384x448 2 boxs, 54.0ms
tensor([[  8.9422, 248.1696,  54.2065, 284.2096]])
tensor([[ 49.1863,   0.0000, 391.9005,  21.6414]])
Speed: 2.0ms preprocess, 